# 04 — Gold Star Schema

Builds consumption-ready star schema from the Silver layer:
- **dim_athlete** — Type 1 (overwrite) from silver.athletes
- **dim_sponsor** — Type 1 from silver.sponsors
- **dim_date** — generated date dimension (2024-2027)
- **fact_deals** — grain: one row per deal, FKs to all dims
- **fact_campaign_performance** — grain: one row per campaign event, FKs to deal + date
- **3 views** for pre-aggregated NIL analytics

In [1]:
from databricks.connect import DatabricksSession
from dotenv import load_dotenv
import os

load_dotenv()

spark = DatabricksSession.builder.serverless().getOrCreate()

UC_CATALOG    = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA     = os.getenv("UC_SCHEMA", "nil_sponsorships")
SILVER_SCHEMA = f"{UC_SCHEMA}_silver"
GOLD_SCHEMA   = f"{UC_SCHEMA}_gold"

print(f"Silver: {UC_CATALOG}.{SILVER_SCHEMA}")
print(f"Gold:   {UC_CATALOG}.{GOLD_SCHEMA}")

Silver: alexander_booth.nil_sponsorships_silver
Gold:   alexander_booth.nil_sponsorships_gold


In [2]:
def add_pk(table: str, name: str, cols: list):
    cat, sch, tbl = table.split(".")
    existing = spark.sql(f"""
        SELECT constraint_name FROM {cat}.information_schema.table_constraints
        WHERE table_schema = '{sch}' AND table_name = '{tbl}'
          AND constraint_type = 'PRIMARY KEY' AND constraint_name = '{name}'
    """).collect()
    if existing:
        print(f"  PK '{name}' already exists on {table}")
        return
    for c in cols:
        try:
            spark.sql(f"ALTER TABLE {table} ALTER COLUMN {c} SET NOT NULL")
        except Exception as e:
            if "ALREADY_NOT_NULLABLE" not in str(e):
                raise
    spark.sql(f"ALTER TABLE {table} ADD CONSTRAINT {name} PRIMARY KEY ({', '.join(cols)}) RELY")
    print(f"  Added PK '{name}' on {table}")


def add_fk(table: str, name: str, cols: list, ref_table: str, ref_cols: list):
    cat, sch, tbl = table.split(".")
    existing = spark.sql(f"""
        SELECT constraint_name FROM {cat}.information_schema.table_constraints
        WHERE table_schema = '{sch}' AND table_name = '{tbl}'
          AND constraint_type = 'FOREIGN KEY' AND constraint_name = '{name}'
    """).collect()
    if existing:
        print(f"  FK '{name}' already exists on {table}")
        return
    spark.sql(f"ALTER TABLE {table} ADD CONSTRAINT {name} FOREIGN KEY ({', '.join(cols)}) REFERENCES {ref_table} ({', '.join(ref_cols)})")
    print(f"  Added FK '{name}' on {table}")

print("Helpers loaded")

Helpers loaded


## dim_date

In [3]:
dim_date = f"{UC_CATALOG}.{GOLD_SCHEMA}.dim_date"

spark.sql(f"DROP TABLE IF EXISTS {dim_date}")

spark.sql(f"""
    CREATE TABLE {dim_date} AS
    WITH dates AS (
        SELECT EXPLODE(SEQUENCE(
            DATE'2024-01-01', DATE'2027-12-31', INTERVAL 1 DAY
        )) AS date_value
    )
    SELECT
        CAST(ROW_NUMBER() OVER (ORDER BY date_value) AS INT) AS date_sk,
        date_value,
        CAST(DATE_FORMAT(date_value, 'yyyyMMdd') AS INT) AS date_key,
        YEAR(date_value)        AS year,
        MONTH(date_value)       AS month,
        DAY(date_value)         AS day,
        QUARTER(date_value)     AS quarter,
        WEEKOFYEAR(date_value)  AS week_of_year,
        DAYOFWEEK(date_value)   AS day_of_week,
        DATE_FORMAT(date_value, 'MMMM')  AS month_name,
        DATE_FORMAT(date_value, 'EEEE')  AS day_name,
        CASE WHEN DAYOFWEEK(date_value) IN (1, 7) THEN TRUE ELSE FALSE END AS is_weekend,
        LAST_DAY(date_value)    AS month_end_date
    FROM dates
""")

add_pk(dim_date, "dim_date_pk", ["date_sk"])

count = spark.table(dim_date).count()
print(f"dim_date: {count:,} rows")

  Added PK 'dim_date_pk' on alexander_booth.nil_sponsorships_gold.dim_date
dim_date: 1,461 rows


## dim_athlete (Type 1)

In [4]:
dim_athlete = f"{UC_CATALOG}.{GOLD_SCHEMA}.dim_athlete"

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {dim_athlete} (
        athlete_sk          STRING NOT NULL,
        athlete_id          STRING,
        first_name          STRING,
        last_name           STRING,
        school              STRING,
        conference          STRING,
        sport               STRING,
        position            STRING,
        year                STRING,
        instagram_followers INT,
        tiktok_followers    INT,
        twitter_followers   INT,
        total_followers     INT
    )
""")

spark.sql(f"""
    INSERT OVERWRITE {dim_athlete}
    SELECT
        athlete_sk, athlete_id, first_name, last_name,
        school, conference, sport, position, year,
        instagram_followers, tiktok_followers, twitter_followers, total_followers
    FROM {UC_CATALOG}.{SILVER_SCHEMA}.athletes
""")

add_pk(dim_athlete, "dim_athlete_pk", ["athlete_sk"])

count = spark.table(dim_athlete).count()
print(f"dim_athlete: {count:,} rows")

  Added PK 'dim_athlete_pk' on alexander_booth.nil_sponsorships_gold.dim_athlete
dim_athlete: 200 rows


## dim_sponsor (Type 1)

In [5]:
dim_sponsor = f"{UC_CATALOG}.{GOLD_SCHEMA}.dim_sponsor"

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {dim_sponsor} (
        sponsor_sk  STRING NOT NULL,
        sponsor_id  STRING,
        brand_name  STRING,
        industry    STRING,
        budget_tier STRING,
        region      STRING
    )
""")

spark.sql(f"""
    INSERT OVERWRITE {dim_sponsor}
    SELECT
        sponsor_sk, sponsor_id, brand_name, industry, budget_tier, region
    FROM {UC_CATALOG}.{SILVER_SCHEMA}.sponsors
""")

add_pk(dim_sponsor, "dim_sponsor_pk", ["sponsor_sk"])

count = spark.table(dim_sponsor).count()
print(f"dim_sponsor: {count:,} rows")

  Added PK 'dim_sponsor_pk' on alexander_booth.nil_sponsorships_gold.dim_sponsor
dim_sponsor: 50 rows


## fact_deals

In [6]:
fact_deals = f"{UC_CATALOG}.{GOLD_SCHEMA}.fact_deals"

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {fact_deals} (
        deal_sk             STRING NOT NULL,
        deal_id             STRING,
        athlete_sk_fk       STRING,
        sponsor_sk_fk       STRING,
        start_date_sk_fk    INT,
        end_date_sk_fk      INT,
        deal_type           STRING,
        deal_amount         DOUBLE,
        status              STRING,
        start_date          DATE,
        end_date            DATE,
        deal_duration_days  INT
    )
    CLUSTER BY (status, start_date_sk_fk)
""")

spark.sql(f"""
    INSERT OVERWRITE {fact_deals}
    SELECT
        d.deal_sk,
        d.deal_id,
        MD5(d.athlete_id)          AS athlete_sk_fk,
        MD5(d.sponsor_id)          AS sponsor_sk_fk,
        ds.date_sk                 AS start_date_sk_fk,
        de.date_sk                 AS end_date_sk_fk,
        d.deal_type,
        d.deal_amount,
        d.status,
        d.start_date,
        d.end_date,
        DATEDIFF(d.end_date, d.start_date) AS deal_duration_days
    FROM {UC_CATALOG}.{SILVER_SCHEMA}.deals d
    LEFT JOIN {UC_CATALOG}.{GOLD_SCHEMA}.dim_date ds ON d.start_date = ds.date_value
    LEFT JOIN {UC_CATALOG}.{GOLD_SCHEMA}.dim_date de ON d.end_date = de.date_value
""")

add_pk(fact_deals, "fact_deals_pk", ["deal_sk"])
add_fk(fact_deals, "fact_deals_athlete_fk",
       ["athlete_sk_fk"], f"{UC_CATALOG}.{GOLD_SCHEMA}.dim_athlete", ["athlete_sk"])
add_fk(fact_deals, "fact_deals_sponsor_fk",
       ["sponsor_sk_fk"], f"{UC_CATALOG}.{GOLD_SCHEMA}.dim_sponsor", ["sponsor_sk"])
add_fk(fact_deals, "fact_deals_start_date_fk",
       ["start_date_sk_fk"], f"{UC_CATALOG}.{GOLD_SCHEMA}.dim_date", ["date_sk"])
add_fk(fact_deals, "fact_deals_end_date_fk",
       ["end_date_sk_fk"], f"{UC_CATALOG}.{GOLD_SCHEMA}.dim_date", ["date_sk"])

count = spark.table(fact_deals).count()
print(f"fact_deals: {count:,} rows")

  Added PK 'fact_deals_pk' on alexander_booth.nil_sponsorships_gold.fact_deals
  Added FK 'fact_deals_athlete_fk' on alexander_booth.nil_sponsorships_gold.fact_deals
  Added FK 'fact_deals_sponsor_fk' on alexander_booth.nil_sponsorships_gold.fact_deals
  Added FK 'fact_deals_start_date_fk' on alexander_booth.nil_sponsorships_gold.fact_deals
  Added FK 'fact_deals_end_date_fk' on alexander_booth.nil_sponsorships_gold.fact_deals
fact_deals: 300 rows


## fact_campaign_performance

In [7]:
fact_campaigns = f"{UC_CATALOG}.{GOLD_SCHEMA}.fact_campaign_performance"

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {fact_campaigns} (
        campaign_sk        STRING NOT NULL,
        campaign_id        STRING,
        deal_sk_fk         STRING,
        date_sk_fk         INT,
        platform           STRING,
        campaign_date      DATE,
        impressions        INT,
        engagements        INT,
        clicks             INT,
        conversions        INT,
        spend_amount       DOUBLE,
        engagement_rate    DOUBLE,
        click_through_rate DOUBLE,
        cost_per_click     DOUBLE,
        cost_per_conversion DOUBLE
    )
    CLUSTER BY (platform, date_sk_fk)
""")

spark.sql(f"""
    INSERT OVERWRITE {fact_campaigns}
    SELECT
        c.campaign_sk,
        c.campaign_id,
        MD5(c.deal_id)              AS deal_sk_fk,
        d.date_sk                   AS date_sk_fk,
        c.platform,
        c.campaign_date,
        c.impressions,
        c.engagements,
        c.clicks,
        c.conversions,
        c.spend_amount,
        c.engagement_rate,
        c.click_through_rate,
        ROUND(c.spend_amount / NULLIF(c.clicks, 0), 2)       AS cost_per_click,
        ROUND(c.spend_amount / NULLIF(c.conversions, 0), 2)  AS cost_per_conversion
    FROM {UC_CATALOG}.{SILVER_SCHEMA}.campaigns c
    LEFT JOIN {UC_CATALOG}.{GOLD_SCHEMA}.dim_date d ON c.campaign_date = d.date_value
""")

add_pk(fact_campaigns, "fact_campaigns_pk", ["campaign_sk"])
add_fk(fact_campaigns, "fact_campaigns_deal_fk",
       ["deal_sk_fk"], f"{UC_CATALOG}.{GOLD_SCHEMA}.fact_deals", ["deal_sk"])
add_fk(fact_campaigns, "fact_campaigns_date_fk",
       ["date_sk_fk"], f"{UC_CATALOG}.{GOLD_SCHEMA}.dim_date", ["date_sk"])

count = spark.table(fact_campaigns).count()
print(f"fact_campaign_performance: {count:,} rows")

  Added PK 'fact_campaigns_pk' on alexander_booth.nil_sponsorships_gold.fact_campaign_performance
  Added FK 'fact_campaigns_deal_fk' on alexander_booth.nil_sponsorships_gold.fact_campaign_performance
  Added FK 'fact_campaigns_date_fk' on alexander_booth.nil_sponsorships_gold.fact_campaign_performance
fact_campaign_performance: 800 rows


## Analytical Views

In [8]:
# Top athletes by total deal value
spark.sql(f"""
    CREATE OR REPLACE VIEW {UC_CATALOG}.{GOLD_SCHEMA}.v_athlete_deal_leaderboard AS
    SELECT
        a.first_name, a.last_name, a.school, a.conference, a.sport,
        a.total_followers,
        COUNT(DISTINCT f.deal_id) AS deal_count,
        SUM(f.deal_amount) AS total_deal_value,
        AVG(f.deal_amount) AS avg_deal_value,
        COUNT(DISTINCT f.sponsor_sk_fk) AS unique_sponsors
    FROM {UC_CATALOG}.{GOLD_SCHEMA}.fact_deals f
    INNER JOIN {UC_CATALOG}.{GOLD_SCHEMA}.dim_athlete a ON f.athlete_sk_fk = a.athlete_sk
    WHERE f.status IN ('Active', 'Completed')
    GROUP BY a.first_name, a.last_name, a.school, a.conference, a.sport, a.total_followers
""")
print("Created: v_athlete_deal_leaderboard")

# Sponsor ROI by industry
spark.sql(f"""
    CREATE OR REPLACE VIEW {UC_CATALOG}.{GOLD_SCHEMA}.v_sponsor_roi AS
    SELECT
        s.brand_name, s.industry, s.budget_tier,
        COUNT(DISTINCT fd.deal_id) AS deal_count,
        SUM(fd.deal_amount) AS total_invested,
        SUM(cp.impressions) AS total_impressions,
        SUM(cp.engagements) AS total_engagements,
        SUM(cp.conversions) AS total_conversions,
        SUM(cp.spend_amount) AS total_campaign_spend,
        ROUND(SUM(cp.impressions) / NULLIF(SUM(cp.spend_amount), 0), 2) AS impressions_per_dollar
    FROM {UC_CATALOG}.{GOLD_SCHEMA}.fact_deals fd
    INNER JOIN {UC_CATALOG}.{GOLD_SCHEMA}.dim_sponsor s ON fd.sponsor_sk_fk = s.sponsor_sk
    LEFT JOIN {UC_CATALOG}.{GOLD_SCHEMA}.fact_campaign_performance cp ON fd.deal_sk = cp.deal_sk_fk
    GROUP BY s.brand_name, s.industry, s.budget_tier
""")
print("Created: v_sponsor_roi")

# Conference deal summary
spark.sql(f"""
    CREATE OR REPLACE VIEW {UC_CATALOG}.{GOLD_SCHEMA}.v_conference_summary AS
    SELECT
        a.conference,
        COUNT(DISTINCT a.athlete_sk) AS athlete_count,
        COUNT(DISTINCT fd.deal_id) AS deal_count,
        SUM(fd.deal_amount) AS total_deal_value,
        AVG(fd.deal_amount) AS avg_deal_value,
        COUNT(DISTINCT fd.sponsor_sk_fk) AS unique_sponsors,
        SUM(cp.impressions) AS total_impressions
    FROM {UC_CATALOG}.{GOLD_SCHEMA}.dim_athlete a
    LEFT JOIN {UC_CATALOG}.{GOLD_SCHEMA}.fact_deals fd ON a.athlete_sk = fd.athlete_sk_fk
    LEFT JOIN {UC_CATALOG}.{GOLD_SCHEMA}.fact_campaign_performance cp ON fd.deal_sk = cp.deal_sk_fk
    GROUP BY a.conference
""")
print("Created: v_conference_summary")

Created: v_athlete_deal_leaderboard
Created: v_sponsor_roi
Created: v_conference_summary


In [9]:
print("=" * 60)
print("GOLD LAYER SUMMARY")
print("=" * 60)

for table in ["dim_date", "dim_athlete", "dim_sponsor",
              "fact_deals", "fact_campaign_performance"]:
    full = f"{UC_CATALOG}.{GOLD_SCHEMA}.{table}"
    count = spark.table(full).count()
    print(f"  {full}: {count:,} rows")

print("\nViews:")
for v in ["v_athlete_deal_leaderboard", "v_sponsor_roi", "v_conference_summary"]:
    print(f"  {UC_CATALOG}.{GOLD_SCHEMA}.{v}")

print("\nSample leaderboard:")
spark.sql(f"""
    SELECT first_name, last_name, school, sport,
           deal_count, total_deal_value, unique_sponsors
    FROM {UC_CATALOG}.{GOLD_SCHEMA}.v_athlete_deal_leaderboard
    ORDER BY total_deal_value DESC
    LIMIT 10
""").show(truncate=False)

print("Gold star schema complete.")

GOLD LAYER SUMMARY
  alexander_booth.nil_sponsorships_gold.dim_date: 1,461 rows
  alexander_booth.nil_sponsorships_gold.dim_athlete: 200 rows
  alexander_booth.nil_sponsorships_gold.dim_sponsor: 50 rows
  alexander_booth.nil_sponsorships_gold.fact_deals: 300 rows
  alexander_booth.nil_sponsorships_gold.fact_campaign_performance: 800 rows

Views:
  alexander_booth.nil_sponsorships_gold.v_athlete_deal_leaderboard
  alexander_booth.nil_sponsorships_gold.v_sponsor_roi
  alexander_booth.nil_sponsorships_gold.v_conference_summary

Sample leaderboard:
+----------+---------+----------------------+------------------+----------+------------------+---------------+
|first_name|last_name|school                |sport             |deal_count|total_deal_value  |unique_sponsors|
+----------+---------+----------------------+------------------+----------+------------------+---------------+
|Jose      |Garcia   |Clemson University    |Men's Basketball  |3         |208482.96000000002|2              |
|Marg